### Import bibliotek

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

### Wczytanie danych

In [2]:
iris = load_iris()
x = iris.data
y = iris.target
df = pd.DataFrame(x, columns=['Długość działki[cm]', 'Szerokość działki[cm]', 'Długość płatka[cm]', 'Szerokość płatka[cm]'])
df['gatunek'] = y

### Analiza danych

In [3]:
print("Liczba wierszy: ", df.shape[0])
print("Liczba cech: ", df.shape[1])
print("Podział na klasy:\n", df['gatunek'].value_counts())
print("Pierwsze 5 danych:\n", df.head())

Liczba wierszy:  150
Liczba cech:  5
Podział na klasy:
 gatunek
0    50
1    50
2    50
Name: count, dtype: int64
Pierwsze 5 danych:
    Długość działki[cm]  Szerokość działki[cm]  Długość płatka[cm]  \
0                  5.1                    3.5                 1.4   
1                  4.9                    3.0                 1.4   
2                  4.7                    3.2                 1.3   
3                  4.6                    3.1                 1.5   
4                  5.0                    3.6                 1.4   

   Szerokość płatka[cm]  gatunek  
0                   0.2        0  
1                   0.2        0  
2                   0.2        0  
3                   0.2        0  
4                   0.2        0  


### Normalizacja

In [4]:
print("Średnie kolumn przed standaryzacją:\n", np.mean(x, axis=0))
print("Odchylenia standardowe kolumn przed standaryzacją:\n", np.std(x, axis=0))
x_srednia = np.mean(x, axis=0)
x_odchylenie_standardowe = np.std(x, axis=0)
x_ustandaryzowane = (x - x_srednia) / x_odchylenie_standardowe
print("Średnie kolumn po standaryzacji:\n", np.mean(x_ustandaryzowane, axis=0))
print("Odchylenia standardowe kolumn po standaryzacji:\n", np.std(x_ustandaryzowane, axis=0))

Średnie kolumn przed standaryzacją:
 [5.84333333 3.05733333 3.758      1.19933333]
Odchylenia standardowe kolumn przed standaryzacją:
 [0.82530129 0.43441097 1.75940407 0.75969263]
Średnie kolumn po standaryzacji:
 [-1.69031455e-15 -1.84297022e-15 -1.69864123e-15 -1.40924309e-15]
Odchylenia standardowe kolumn po standaryzacji:
 [1. 1. 1. 1.]


### Utworzenie neuronów

In [5]:
neurony = np.random.randn(3, 4)

### Implementacja odległości

In [6]:
def oblicz_odleglosci(kwiatek, neurony):
    odleglosci_d = np.sqrt(np.sum((neurony - kwiatek) ** 2, axis=1))
    return odleglosci_d

### Znalezienie neuronu zwycięskiego

In [7]:
def znajdz_zwyciezce(kwiatek, neurony):
    indeks_zwyciezcy = np.argmin(oblicz_odleglosci(kwiatek, neurony))
    return indeks_zwyciezcy

### Implementacja aktualizacji wag

In [8]:
def przybliz_neuron(kwiatek, neurony, indeks_zwyciezcy, krok):
    neurony[indeks_zwyciezcy] = neurony[indeks_zwyciezcy] + krok * (kwiatek - neurony[indeks_zwyciezcy])
    return neurony

### Ostateczna funkcja uproszczonego algorytmu SOM
Zastosowania z lab'u 2:
- zmniejszanie współczynnika uczenia w miarę trwania algorytmu
- wyświetlanie wartości współczynnika co 10 epok

In [9]:
def algorytm_som(przyklady, neurony, tempo_uczenia, liczba_epok):
    T = liczba_epok
    while liczba_epok != 0:
        tempo = tempo_uczenia * (liczba_epok / T)

        if liczba_epok % 10 == 0:
            print("Wartość współczynnika: ", tempo)

        for przyklad in przyklady:
            zwyciezca = znajdz_zwyciezce(przyklad, neurony)
            neurony = przybliz_neuron(przyklad, neurony, zwyciezca, tempo)

        liczba_epok -= 1

    return neurony

### Testy
Zgodnie z poleceniem w lab'ie 2-gim:
- przypisanie przykładów do każdego neuronu

In [10]:
wytrenowane_neurony = algorytm_som(x_ustandaryzowane, neurony.copy(), tempo_uczenia=0.5, liczba_epok=100)
przypisane_etykiety = [znajdz_zwyciezce(kwiatek, wytrenowane_neurony) for kwiatek in x_ustandaryzowane]
ilosci_elementow = {0: 0, 1: 0, 2: 0}
for etykieta in przypisane_etykiety:
    ilosci_elementow[etykieta] += 1

print("WYNIKI UCZENIA ALGORYTMEM SOM")
print("Rozkład kwiatków w grupach:")
for neuron_i, ilosc in ilosci_elementow.items():
    print(f"Neuron {neuron_i + 1} zgrupował {ilosc} sztuk")
print("\nOstateczne wagi neuronów (pozycje w 4-wymiarowej przestrzeni):")
print(np.round(wytrenowane_neurony, 2))

Wartość współczynnika:  0.5
Wartość współczynnika:  0.45
Wartość współczynnika:  0.4
Wartość współczynnika:  0.35
Wartość współczynnika:  0.3
Wartość współczynnika:  0.25
Wartość współczynnika:  0.2
Wartość współczynnika:  0.15
Wartość współczynnika:  0.1
Wartość współczynnika:  0.05
WYNIKI UCZENIA ALGORYTMEM SOM
Rozkład kwiatków w grupach:
Neuron 1 zgrupował 54 sztuk
Neuron 2 zgrupował 50 sztuk
Neuron 3 zgrupował 46 sztuk

Ostateczne wagi neuronów (pozycje w 4-wymiarowej przestrzeni):
[[-0.04 -0.87  0.38  0.33]
 [-1.02  0.84 -1.3  -1.25]
 [ 1.15  0.09  1.02  1.07]]


### [LAB 2] Macierz rozkładu przykładów

In [11]:
macierz_wynikow = np.zeros((3, 3), dtype=int)
for nazwa_gatunku, przypisany_neuron in zip(y, przypisane_etykiety):
    macierz_wynikow[przypisany_neuron][nazwa_gatunku] += 1
print("\nMacierz rozkładu przykładów")
print("Neuron | Setosa | Versicolor | Virginica")
print("-" * 45)
for i in range(3):
    print(f"N{i+1}     | {macierz_wynikow[i][0]}     | {macierz_wynikow[i][1]}         | {macierz_wynikow[i][2]}")



Macierz rozkładu przykładów
Neuron | Setosa | Versicolor | Virginica
---------------------------------------------
N1     | 0     | 39         | 15
N2     | 50     | 0         | 0
N3     | 0     | 11         | 35
